# 01 Data Collection

In this notebook, we collect historical Formula 1 race data from 2018 onward.

The goal of this step is to create a clean raw dataset that will later be used for:

- feature engineering
- machine learning model training
- race result prediction
- driver performance analysis
- driver profile cards

We will use the Jolpica F1 API, which provides Formula 1 data in an Ergast-compatible format.

# Import Libs

In [ ]:
import requests
import pandas as pd
import time 
from pathlib import Path

# Create Project Paths

Save collected data in data/raw

In [ ]:
PROJECT_ROOT = Path("..")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Set API Parametrs 

define URL of API and seasons we want collect

In [ ]:
BASE_URL = "https://api.jolpi.ca/ergast/f1"

START_YEAR = 2018
END_YEAR = 2025

SEASONS = list(range(START_YEAR, END_YEAR +1))

SEASONS

# Create API request function

get url, return json

In [ ]:
import requests
import time

def get_json(url, max_retries=5, timeout=60):
    """
    Send GET request to the API with retries.
    
    If the API is slow or temporarily unavailable, the function waits
    and tries again instead of stopping the whole notebook.
    """
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Request attempt {attempt}: {url}")
            
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            
            return response.json()
        
        except requests.exceptions.Timeout:
            print(f"Timeout on attempt {attempt}. Waiting before retry...")
            time.sleep(5)
        
        except requests.exceptions.RequestException as error:
            print(f"Request failed on attempt {attempt}")
            print(f"Error: {error}")
            time.sleep(5)
    
    print(f"Failed after {max_retries} attempts: {url}")
    return None

# Test API Connection

In [ ]:
test_url = f"{BASE_URL}/2018/1/results.json"
test_data = get_json(test_url)

test_data.keys() if test_data else None

In [ ]:
test_data["MRData"].keys()

In [ ]:
test_data["MRData"]["RaceTable"].keys()

In [ ]:
len(test_data["MRData"]["RaceTable"]["Races"])

In [ ]:
def get_season_rounds(season):
    """
    Get number of races in a season.
    """
    url = f"{BASE_URL}/{season}.json"
    data = get_json(url)
    
    if data is None:
        return []
    
    races = data["MRData"]["RaceTable"]["Races"]
    rounds = [int(race["round"]) for race in races]
    
    return rounds

In [ ]:
rounds_2018 = get_season_rounds(2018)
rounds_2018

## 6. Collect Race Results

Now we collect race results for each season.

For every race, we extract:

- season
- round
- race name
- race date
- circuit information
- driver information
- constructor information
- grid position
- final position
- points
- laps
- race status

This table will become the base dataset for our machine learning project.

In [ ]:
def collect_race_results(seasons):
    """
    Collect Formula 1 race results race by race.
    
    This approach is more stable than requesting the whole season at once.
    """
    
    all_results = []
    
    for season in seasons:
        print(f"\nCollecting season {season}...")
        
        rounds = get_season_rounds(season)
        
        if not rounds:
            print(f"No rounds found for season {season}")
            continue
        
        for round_number in rounds:
            print(f"Collecting season {season}, round {round_number}...")
            
            url = f"{BASE_URL}/{season}/{round_number}/results.json"
            data = get_json(url)
            
            if data is None:
                continue
            
            races = data["MRData"]["RaceTable"]["Races"]
            
            if len(races) == 0:
                continue
            
            race = races[0]
            
            season_year = int(race["season"])
            round_number = int(race["round"])
            race_name = race["raceName"]
            race_date = race.get("date")
            
            circuit = race["Circuit"]
            circuit_id = circuit["circuitId"]
            circuit_name = circuit["circuitName"]
            circuit_country = circuit["Location"]["country"]
            circuit_locality = circuit["Location"]["locality"]
            
            for result in race["Results"]:
                driver = result["Driver"]
                constructor = result["Constructor"]
                
                row = {
                    "season": season_year,
                    "round": round_number,
                    "race_name": race_name,
                    "race_date": race_date,
                    
                    "circuit_id": circuit_id,
                    "circuit_name": circuit_name,
                    "circuit_country": circuit_country,
                    "circuit_locality": circuit_locality,
                    
                    "driver_id": driver["driverId"],
                    "driver_code": driver.get("code"),
                    "driver_number": driver.get("permanentNumber"),
                    "driver_given_name": driver["givenName"],
                    "driver_family_name": driver["familyName"],
                    "driver_nationality": driver["nationality"],
                    
                    "constructor_id": constructor["constructorId"],
                    "constructor_name": constructor["name"],
                    "constructor_nationality": constructor["nationality"],
                    
                    "grid": result.get("grid"),
                    "position": result.get("position"),
                    "position_text": result.get("positionText"),
                    "points": result.get("points"),
                    "laps": result.get("laps"),
                    "status": result.get("status")
                }
                
                all_results.append(row)
            
            time.sleep(1)
    
    return pd.DataFrame(all_results)

In [ ]:
SEASONS_TEST = [2018]

race_results_df = collect_race_results(SEASONS_TEST)

race_results_df.head()

In [ ]:
race_results_df.shape

# Save collected data

In [ ]:
race_results_df.to_csv("../data/raw/race_results_2018_test.csv", index=False)

# Collect data from other years

In [ ]:
SEASONS = list(range(2018, 2026))

race_results_df = collect_race_results(SEASONS)

race_results_df.head()

# Save results from 2018 - 2026

In [ ]:
race_results_path = "../data/raw/race_results_2018_2026.csv"

race_results_df.to_csv(race_results_path, index=False)

race_results_check = pd.read_csv(race_results_path)

race_results_check.head()

# Check values

In [ ]:
race_results_check.shape

In [ ]:
race_results_check.info()

In [ ]:
race_results_check.groupby("season")["race_name"].nunique()

In [ ]:
race_results_check.groupby(["season", "race_name"]).size().head(30)

# Check missing races

In [ ]:
races_per_season = (
    race_results_check
    .groupby("season")["race_name"]
    .nunique()
    .reset_index(name="number_of_races")
)

races_per_season

In [ ]:
races_per_season.sort_values("season")

## Check Number of Drivers per Race

In [ ]:
drivers_per_race = (
    race_results_check
    .groupby(["season", "round", "race_name"])["driver_id"]
    .nunique()
    .reset_index(name="number_of_drivers")
)

drivers_per_race.head()

In [ ]:
drivers_per_race[drivers_per_race["number_of_drivers"] < 15]

# Check Duplicates

In [ ]:
duplicates = race_results_check.duplicated(
    subset=["season", "round", "driver_id"],
    keep=False
)

race_results_check[duplicates]

In [ ]:
race_results_check.isnull().sum()

In [ ]:
important_columns = [
    "season",
    "round",
    "race_name",
    "circuit_id",
    "driver_id",
    "constructor_id",
    "grid",
    "position",
    "points",
    "status"
]

race_results_check[important_columns].isnull().sum()

In [ ]:
numeric_columns = ["season", "round", "grid", "position", "points", "laps"]

for col in numeric_columns:
    race_results_check[col] = pd.to_numeric(race_results_check[col], errors="coerce")

race_results_check.dtypes

# Save validated raw version

In [ ]:
validated_race_results_path = RAW_DATA_DIR / "race_results_2018_2026_validated.csv"

race_results_check.to_csv(validated_race_results_path, index=False)

validated_race_results_path

# Collect Qualifying Results Race by Race

In [ ]:
def collect_qualifying_results(seasons):
    """
    Collect Formula 1 qualifying results race by race.
    This method is more stable than collecting the whole season at once.
    """
    
    all_qualifying = []
    
    for season in seasons:
        print(f"\nCollecting qualifying results for season {season}...")
        
        rounds = get_season_rounds(season)
        
        if not rounds:
            print(f"No rounds found for season {season}")
            continue
        
        for round_number in rounds:
            print(f"Collecting qualifying: season {season}, round {round_number}...")
            
            url = f"{BASE_URL}/{season}/{round_number}/qualifying.json"
            data = get_json(url)
            
            if data is None:
                continue
            
            races = data["MRData"]["RaceTable"]["Races"]
            
            if len(races) == 0:
                continue
            
            race = races[0]
            
            season_year = int(race["season"])
            round_number = int(race["round"])
            race_name = race["raceName"]
            
            circuit = race["Circuit"]
            circuit_id = circuit["circuitId"]
            circuit_name = circuit["circuitName"]
            
            for result in race["QualifyingResults"]:
                driver = result["Driver"]
                constructor = result["Constructor"]
                
                row = {
                    "season": season_year,
                    "round": round_number,
                    "race_name": race_name,
                    "circuit_id": circuit_id,
                    "circuit_name": circuit_name,
                    
                    "driver_id": driver["driverId"],
                    "driver_code": driver.get("code"),
                    "driver_name": driver["givenName"] + " " + driver["familyName"],
                    
                    "constructor_id": constructor["constructorId"],
                    "constructor_name": constructor["name"],
                    
                    "qualifying_position": result.get("position"),
                    "q1": result.get("Q1"),
                    "q2": result.get("Q2"),
                    "q3": result.get("Q3")
                }
                
                all_qualifying.append(row)
            
            time.sleep(1)
    
    return pd.DataFrame(all_qualifying)

In [ ]:
qualifying_test_df = collect_qualifying_results([2018])

qualifying_test_df.head()

In [ ]:
qualifying_test_df.shape

In [ ]:
qualifying_df = collect_qualifying_results(SEASONS)

qualifying_df.head()

# Save qualifying data

In [ ]:
qualifying_df["qualifying_position"] = pd.to_numeric(
    qualifying_df["qualifying_position"],
    errors="coerce"
)

qualifying_path = RAW_DATA_DIR / "qualifying_results_2018_2026.csv"

qualifying_df.to_csv(qualifying_path, index=False)

qualifying_path

# Collect Race Schedule and Circuit Data

In [ ]:
def collect_race_schedule(seasons):
    """
    Collect race schedule and circuit information for multiple seasons.
    
    This function collects:
    - season
    - round
    - race name
    - race date
    - circuit id
    - circuit name
    - circuit location
    - country
    
    Parameters:
        seasons (list): list of seasons
        
    Returns:
        pandas.DataFrame: race schedule and circuit data
    """
    
    all_races = []
    
    for season in seasons:
        print(f"\nCollecting race schedule for season {season}...")
        
        url = f"{BASE_URL}/{season}.json"
        data = get_json(url)
        
        if data is None:
            print(f"No data for season {season}")
            continue
        
        races = data["MRData"]["RaceTable"]["Races"]
        
        for race in races:
            circuit = race["Circuit"]
            location = circuit["Location"]
            
            row = {
                "season": int(race["season"]),
                "round": int(race["round"]),
                "race_name": race["raceName"],
                "race_date": race.get("date"),
                "race_time": race.get("time"),
                
                "circuit_id": circuit["circuitId"],
                "circuit_name": circuit["circuitName"],
                "circuit_url": circuit.get("url"),
                
                "locality": location.get("locality"),
                "country": location.get("country"),
                "lat": location.get("lat"),
                "long": location.get("long")
            }
            
            all_races.append(row)
        
        time.sleep(1)
    
    return pd.DataFrame(all_races)

In [ ]:
schedule_df = collect_race_schedule(SEASONS)

schedule_df.head()

In [ ]:
schedule_path = RAW_DATA_DIR / "race_schedule_2018_2026.csv"

schedule_df.to_csv(schedule_path, index=False)

schedule_path